In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

chunks_df = pd.read_csv(PROCESSED_DIR / "chunks.csv")
embeddings = np.load(PROCESSED_DIR / "embeddings.npy")

print("Chunks:", len(chunks_df))
print("Embeddings:", embeddings.shape)

Chunks: 27
Embeddings: (27, 384)


In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

def retrieve(query, top_k=3):
    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    top_indices = similarities.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "source": chunks_df.iloc[index]["source"],
            "chunk_id": int(chunks_df.iloc[index]["chunk_id"]),
            "similarity": float(similarities[index]),
            "text": chunks_df.iloc[index]["text"]
        })

    return results

d:\ai-customer-support-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6867.27it/s]


In [4]:
query = "My payment failed. What should I do?"

results = retrieve(query, top_k=3)

for result in results:
    print("=" * 70)
    print("SOURCE:", result["source"])
    print("SIMILARITY:", round(result["similarity"], 4))
    print(result["text"])

SOURCE: payments.md
SIMILARITY: 0.6506
# Payment Support

## Payment Failed

If a payment fails, customers should verify that their payment method has sufficient funds and that the billing information is correct.

Customers can try the payment again after checking their payment details. If the payment continues to fail, they should contact customer support.

## Payment Declined

A payment may be declined by the payment provider because of insufficient funds, incorrect payment information, or security restrictions.

Customers should v
SOURCE: payments.md
SIMILARITY: 0.5673
g.

If the payment remains pending for an extended period, they should contact customer support with the transaction details.

## Payment Reversed

A payment may be reversed if the transaction could not be completed successfully. Customers should check their payment history to confirm the current status.

If the amount has not been returned after the expected processing period, customers should contact support.

## Pa

In [ ]:
context = "\n\n".join(
    [
        f"Source: {result['source']}\n{result['text']}"
        for result in results
    ]
)

Source: payments.md
# Payment Support

## Payment Failed

If a payment fails, customers should verify that their payment method has sufficient funds and that the billing information is correct.

Customers can try the payment again after checking their payment details. If the payment continues to fail, they should contact customer support.

## Payment Declined

A payment may be declined by the payment provider because of insufficient funds, incorrect payment information, or security restrictions.

Customers should v

Source: payments.md
g.

If the payment remains pending for an extended period, they should contact customer support with the transaction details.

## Payment Reversed

A payment may be reversed if the transaction could not be completed successfully. Customers should check their payment history to confirm the current status.

If the amount has not been returned after the expected processing period, customers should contact support.

## Payment Method

Customers can use the p

In [6]:
prompt = f"""
You are a customer support assistant.

Answer the customer's question using only the information provided
in the context below.

If the context does not contain enough information to answer,
say that you do not have enough information.

Customer question:
{query}

Context:
{context}

Answer:
"""

print(prompt)


You are a customer support assistant.

Answer the customer's question using only the information provided
in the context below.

If the context does not contain enough information to answer,
say that you do not have enough information.

Customer question:
My payment failed. What should I do?

Context:
Source: payments.md
# Payment Support

## Payment Failed

If a payment fails, customers should verify that their payment method has sufficient funds and that the billing information is correct.

Customers can try the payment again after checking their payment details. If the payment continues to fail, they should contact customer support.

## Payment Declined

A payment may be declined by the payment provider because of insufficient funds, incorrect payment information, or security restrictions.

Customers should v

Source: payments.md
g.

If the payment remains pending for an extended period, they should contact customer support with the transaction details.

## Payment Reversed

A paym

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

In [8]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    token=hf_token
)

print("Hugging Face client created successfully.")

Hugging Face client created successfully.


In [17]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

print("Model:", MODEL_ID)

Model: meta-llama/Llama-3.1-8B-Instruct


In [18]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful customer support assistant. "
            "Answer only using the provided context. "
            "If the context does not contain enough information, "
            "say that you do not have enough information."
        )
    },
    {
        "role": "user",
        "content": f"""
Customer question:
{query}

Context:
{context}

Answer the customer clearly and concisely.
"""
    }
]

response = client.chat_completion(
    messages=messages,
    model=MODEL_ID,
    max_tokens=200,
    temperature=0.2
)

answer = response.choices[0].message.content

print(answer)

I'm sorry to hear that your payment failed. To resolve the issue, please check that your payment method has sufficient funds and that your billing information is correct. If you've already done this and the payment still fails, you can try making the payment again. If it continues to fail, please contact our customer support team so we can assist you further.


In [19]:
sources = list(dict.fromkeys(
    result["source"] for result in results
))

print("Answer:")
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Answer:
I'm sorry to hear that your payment failed. To resolve the issue, please check that your payment method has sufficient funds and that your billing information is correct. If you've already done this and the payment still fails, you can try making the payment again. If it continues to fail, please contact our customer support team so we can assist you further.

Sources:
- payments.md


In [20]:
query = "I forgot my password. How can I reset it?"

results = retrieve(query, top_k=3)

context = "\n\n".join(
    [
        f"Source: {result['source']}\n{result['text']}"
        for result in results
    ]
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful customer support assistant. "
            "Answer only using the provided context. "
            "If the context does not contain enough information, "
            "say that you do not have enough information."
        )
    },
    {
        "role": "user",
        "content": f"""
Customer question:
{query}

Context:
{context}

Answer the customer clearly and concisely.
"""
    }
]

response = client.chat_completion(
    messages=messages,
    model=MODEL_ID,
    max_tokens=200,
    temperature=0.2
)

answer = response.choices[0].message.content

sources = list(dict.fromkeys(
    result["source"] for result in results
))

print("Answer:")
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Answer:
To reset your password, please use the password reset option on the sign-in page. A password reset link will be sent to the registered email address. If you don't receive the email, check your spam or junk folder and make sure you entered the correct email address. If you still can't receive the email, contact customer support for assistance.

Sources:
- account.md
- technical_support.md


In [21]:
query = "Can I change my delivery address after my order has shipped?"

results = retrieve(query, top_k=3)

context = "\n\n".join(
    [
        f"Source: {result['source']}\n{result['text']}"
        for result in results
    ]
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful customer support assistant. "
            "Answer only using the provided context. "
            "Do not make up information. "
            "If the context does not contain enough information, "
            "say that you do not have enough information."
        )
    },
    {
        "role": "user",
        "content": f"""
Customer question:
{query}

Context:
{context}

Answer the customer clearly and concisely.
"""
    }
]

response = client.chat_completion(
    messages=messages,
    model=MODEL_ID,
    max_tokens=200,
    temperature=0.2
)

answer = response.choices[0].message.content

print("Answer:")
print(answer)

Answer:
Unfortunately, I don't have enough information to determine if you can change your delivery address after your order has shipped. The provided context only contains information about refunds, returns, damaged products, and account management, but does not mention changing delivery addresses. If you could provide more context or details about your order, I may be able to assist you better.


In [22]:
rag_result = {
    "query": query,
    "answer": answer,
    "sources": sources if "sources" in locals() else []
}

print("RAG experiment completed.")
print("Question:", rag_result["query"])
print("Answer:", rag_result["answer"])

RAG experiment completed.
Question: Can I change my delivery address after my order has shipped?
Answer: Unfortunately, I don't have enough information to determine if you can change your delivery address after your order has shipped. The provided context only contains information about refunds, returns, damaged products, and account management, but does not mention changing delivery addresses. If you could provide more context or details about your order, I may be able to assist you better.


In [23]:
import json

rag_results_path = PROCESSED_DIR / "rag_test_result.json"

with open(rag_results_path, "w", encoding="utf-8") as f:
    json.dump(rag_result, f, indent=4, ensure_ascii=False)

print("Saved:", rag_results_path)

Saved: d:\ai-customer-support-assistant\data\processed\rag_test_result.json
